# Анализ и визуализация данных об автомобилях

В этом ноутбуке используются три CSV-файла по теме проекта **«Автомобили: модели, производители и электромобили»**:

- `car_models.csv` — автомобильные модели, производители, год появления и тип модели;
- `car_manufacturers.csv` — автопроизводители, страны, штаб-квартиры и год основания;
- `electric_cars.csv` — электромобили, производители и год появления.

Цель анализа — сравнить производителей, страны происхождения автопроизводителей, годы появления моделей и развитие сегмента электромобилей.

## 1. Импорт библиотек

In [ ]:
# подключаем библиотеки для работы с таблицами и графиками
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# настраиваем внешний вид графиков
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)

## 2. Настройка ссылки на репозиторий

In [ ]:
# укажи владельца и название репозитория
repo_owner = "USERNAME"
repo_name = "REPOSITORY_NAME"

# собираем базовую ссылку на папку data
base_url = f"https://raw.githubusercontent.com/{repo_owner}/{repo_name}/main/data"

print(base_url)

В ячейке выше нужно заменить `USERNAME` и `REPOSITORY_NAME` на реальные значения из ссылки GitHub.

Например, если ссылка на репозиторий выглядит так:

`https://github.com/vasilevsavelij135-spec/python-ai-Karmanov-Vasiliy`

то значения будут такими:

`repo_owner = "vasilevsavelij135-spec"`

`repo_name = "python-ai-Karmanov-Vasiliy"`

## 3. Загрузка данных

In [ ]:
# указываем raw-ссылки на csv-файлы из github
models_url = f"{base_url}/car_models.csv"
manufacturers_url = f"{base_url}/car_manufacturers.csv"
electric_url = f"{base_url}/electric_cars.csv"

# читаем csv-файлы
df_models = pd.read_csv(models_url)
df_manufacturers = pd.read_csv(manufacturers_url)
df_electric = pd.read_csv(electric_url)

# выводим размеры таблиц
print(f"car_models: {df_models.shape[0]} строк, {df_models.shape[1]} столбцов")
print(f"car_manufacturers: {df_manufacturers.shape[0]} строк, {df_manufacturers.shape[1]} столбцов")
print(f"electric_cars: {df_electric.shape[0]} строк, {df_electric.shape[1]} столбцов")

# показываем первые строки
display(df_models.head())
display(df_manufacturers.head())
display(df_electric.head())

## 4. Подготовка данных

In [ ]:
# переименовываем столбцы в таблице моделей автомобилей
df_models = df_models.rename(columns={
    "model": "model_url",
    "modelLabel": "model",
    "manufacturer": "manufacturer_url",
    "manufacturerLabel": "manufacturer",
    "inception": "inception",
    "inceptionYear": "inception_year",
    "modelTypeLabel": "model_type"
})

# переименовываем столбцы в таблице производителей
df_manufacturers = df_manufacturers.rename(columns={
    "manufacturer": "manufacturer_url",
    "manufacturerLabel": "manufacturer",
    "country": "country_url",
    "countryLabel": "country",
    "headquarters": "headquarters_url",
    "headquartersLabel": "headquarters",
    "inception": "inception",
    "foundingYear": "founding_year"
})

# переименовываем столбцы в таблице электромобилей
df_electric = df_electric.rename(columns={
    "car": "car_url",
    "carLabel": "car",
    "manufacturer": "manufacturer_url",
    "manufacturerLabel": "manufacturer",
    "inception": "inception",
    "inceptionYear": "inception_year",
    "typeLabel": "type"
})

# преобразуем годы к числовому типу
df_models["inception_year"] = pd.to_numeric(df_models["inception_year"], errors="coerce")
df_manufacturers["founding_year"] = pd.to_numeric(df_manufacturers["founding_year"], errors="coerce")
df_electric["inception_year"] = pd.to_numeric(df_electric["inception_year"], errors="coerce")

# преобразуем даты к datetime
df_models["inception"] = pd.to_datetime(df_models["inception"], errors="coerce", utc=True)
df_manufacturers["inception"] = pd.to_datetime(df_manufacturers["inception"], errors="coerce", utc=True)
df_electric["inception"] = pd.to_datetime(df_electric["inception"], errors="coerce", utc=True)

# проверяем результат
display(df_models.head())
display(df_manufacturers.head())
display(df_electric.head())

## График 1. Топ-10 производителей по количеству моделей

In [ ]:
# считаем количество моделей по производителям
top_model_manufacturers = (
    df_models["manufacturer"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)
top_model_manufacturers.columns = ["manufacturer", "model_count"]

# строим горизонтальную диаграмму
plt.figure(figsize=(11, 6))
sns.barplot(data=top_model_manufacturers, x="model_count", y="manufacturer")

plt.title("Топ-10 производителей по количеству моделей")
plt.xlabel("Количество моделей")
plt.ylabel("Производитель")

plt.tight_layout()
plt.show()

**Вывод:**  
На графике видно, какие производители чаще всего встречаются среди автомобильных моделей в датасете. Это помогает определить компании с наиболее широким модельным рядом или с лучшей представленностью в Wikidata.

## График 2. Страны по количеству автопроизводителей

In [ ]:
# считаем количество автопроизводителей по странам
top_countries = (
    df_manufacturers["country"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)
top_countries.columns = ["country", "manufacturer_count"]

# строим диаграмму
plt.figure(figsize=(11, 6))
sns.barplot(data=top_countries, x="manufacturer_count", y="country")

plt.title("Топ-10 стран по количеству автопроизводителей")
plt.xlabel("Количество автопроизводителей")
plt.ylabel("Страна")

plt.tight_layout()
plt.show()

**Вывод:**  
График показывает, какие страны чаще всего представлены среди автопроизводителей. Такие страны можно рассматривать как важные центры автомобильной индустрии.

## График 3. Распределение автомобильных моделей по годам появления

In [ ]:
# оставляем годы в разумном диапазоне для автомобильной индустрии
model_years = df_models[
    (df_models["inception_year"] >= 1880) &
    (df_models["inception_year"] <= 2026)
]["inception_year"].dropna()

# строим гистограмму
plt.figure(figsize=(12, 6))
sns.histplot(model_years, bins=30, kde=True)

plt.title("Распределение автомобильных моделей по годам появления")
plt.xlabel("Год появления модели")
plt.ylabel("Количество моделей")

plt.tight_layout()
plt.show()

**Вывод:**  
Гистограмма показывает, в какие периоды в данных чаще появляются новые автомобильные модели. Это помогает увидеть временные пики развития и расширения модельных рядов.

## График 4. Динамика появления электромобилей по годам

In [ ]:
# группируем электромобили по году появления
electric_by_year = (
    df_electric[
        (df_electric["inception_year"] >= 1990) &
        (df_electric["inception_year"] <= 2026)
    ]
    .groupby("inception_year")
    .size()
    .reset_index(name="car_count")
)

# строим линейный график
plt.figure(figsize=(12, 6))
sns.lineplot(data=electric_by_year, x="inception_year", y="car_count", marker="o")

plt.title("Динамика появления электромобилей по годам")
plt.xlabel("Год")
plt.ylabel("Количество электромобилей")

plt.tight_layout()
plt.show()

**Вывод:**  
Линейный график показывает, как менялось количество электромобилей в данных. Если после 2000-х или 2010-х годов наблюдается рост, это отражает усиление интереса к электрическому транспорту.

## График 5. Топ-10 производителей электромобилей

In [ ]:
# считаем количество электромобилей по производителям
top_electric_manufacturers = (
    df_electric["manufacturer"]
    .dropna()
    .value_counts()
    .head(10)
    .reset_index()
)
top_electric_manufacturers.columns = ["manufacturer", "electric_count"]

# строим диаграмму
plt.figure(figsize=(11, 6))
sns.barplot(data=top_electric_manufacturers, x="electric_count", y="manufacturer")

plt.title("Топ-10 производителей электромобилей")
plt.xlabel("Количество электромобилей")
plt.ylabel("Производитель")

plt.tight_layout()
plt.show()

**Вывод:**  
График показывает, какие компании чаще всего представлены среди электромобилей. Это позволяет сравнить традиционных производителей и компании, которые сильнее связаны с электрическим транспортом.

## График 6. Общее число моделей и число электромобилей по производителям

In [ ]:
# считаем общее число моделей по производителям
total_models = df_models.groupby("manufacturer").size().reset_index(name="total_models")

# считаем число электромобилей по производителям
electric_models = df_electric.groupby("manufacturer").size().reset_index(name="electric_models")

# объединяем данные
manufacturer_compare = total_models.merge(electric_models, on="manufacturer", how="left")
manufacturer_compare["electric_models"] = manufacturer_compare["electric_models"].fillna(0)

# выбираем производителей с наибольшим числом моделей
manufacturer_compare = manufacturer_compare.sort_values("total_models", ascending=False).head(10)

# переводим таблицу в длинный формат для графика
manufacturer_compare_long = manufacturer_compare.melt(
    id_vars="manufacturer",
    value_vars=["total_models", "electric_models"],
    var_name="model_type",
    value_name="count"
)

# строим сгруппированную столбчатую диаграмму
plt.figure(figsize=(12, 6))
sns.barplot(data=manufacturer_compare_long, x="manufacturer", y="count", hue="model_type")

plt.title("Общее число моделей и число электромобилей по производителям")
plt.xlabel("Производитель")
plt.ylabel("Количество моделей")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Тип подсчёта")

plt.tight_layout()
plt.show()

**Вывод:**  
Сравнение показывает, какую часть модельного ряда производителей составляют электромобили. У некоторых компаний электромобилей может быть мало относительно общего числа моделей, а у других они занимают более заметное место.

## Общий вывод

В этом ноутбуке были проанализированы три набора данных по автомобильной индустрии: модели автомобилей, производители и электромобили.

Анализ показал, какие производители чаще всего представлены в данных, какие страны чаще встречаются среди автопроизводителей, а также как распределены годы появления автомобильных моделей.

Отдельное внимание было уделено электромобилям. Графики позволяют увидеть динамику появления электромобилей по годам и сравнить производителей по числу электрических моделей.

Данные подходят для анализа, потому что содержат как категориальные признаки, например модель, производитель, страна и тип автомобиля, так и числовые признаки, связанные с годами появления моделей и основания производителей. Это позволяет изучать связь между производителем, страной и развитием электромобильного сегмента.